# 기계학습특론 10주차 실습 — 딥러닝 분류모델 (PyTorch 개정판)

**주제: 뇌 MRI 4-클래스 종양 분류 (glioma / meningioma / no_tumor / pituitary)**

이 노트북은 기존 실습(`1.mri/main.py`, TensorFlow 2.13 + tensorflow-addons)을 **PyTorch 2.x** 로 전환하고,
10주차 강의자료(PART 1~6)의 내용을 순서대로 실습하도록 재구성한 것입니다.

| 노트북 단계 | 강의자료 대응 |
|---|---|
| 1. 데이터 점검 · 누수 확인 | PART 5 (평가 프로토콜과 데이터 누수) |
| 2. Dataset / DataLoader / 증강 | PART 1·3 (텐서, RandAugment·Mixup) |
| 3. 기준선 SimpleCNN | PART 2 (CNN 기본 요소) |
| 4. 학습 루프 (AdamW · 코사인 · AMP) | PART 3 (학습 안정화 기법) |
| 5. 평가 (macro-F1 · 혼동행렬 · AUROC) | PART 2 (평가), PART 5 (불균형) |
| 6. ResNet-18 직접 구현 | PART 4 (잔차 연결) |
| 7. 전이학습 (ResNet-50 / ConvNeXt-T) | PART 4·5 |
| 8. 클래스 불균형 처리 | PART 5 |
| 9. 캘리브레이션 (Temperature scaling · ECE) | PART 5 |
| 10. Grad-CAM 해석 | PART 5 |
| 11. 결과 정리 · 과제 | PART 6 |

---

## 기존 `main.py` 에서 발견된 문제와 이번 개정에서의 수정

| # | 기존 코드의 문제 | 영향 | 이번 노트북의 처리 |
|---|---|---|---|
| 1 | `X = X / 255.` 로 정규화한 뒤 모델 안에서 다시 `layers.Rescaling(1.0/255)` | 입력이 0~0.0039 범위로 **이중 축소** → 학습 불안정 | 정규화를 전처리 한 곳에서만 수행 (ImageNet 평균·표준편차) |
| 2 | `EarlyStopping(monitor='val_loss', mode='max')` | 검증 **손실이 커질 때** 좋은 것으로 판단 → 조기 종료가 반대로 동작 | 검증 macro-F1 **최대화** 기준으로 체크포인트 저장 |
| 3 | `LabelEncoder` 를 train/val/test 에 각각 `fit` | 마지막 `fit` 만 남아 의미 없음 (여기서는 라벨이 이미 0~3 이라 우연히 무해) | 클래스→인덱스 사상을 한 번만 정의하고 공유 |
| 4 | 모든 이미지를 `np.array` 로 메모리에 적재 | 2,870장 × 256×256×3 × float32 ≈ **2.2 GB** (테스트까지 포함하면 더 큼) | `Dataset` 으로 지연 로딩 + `DataLoader` 병렬 디코딩 |
| 5 | 데이터 증강 없음 | 소규모 의료 영상에서 과적합 | Crop/Flip/Rotation/RandAugment + (선택) Mixup·CutMix |
| 6 | `tensorflow-addons` 의존 (2024년 개발 종료) | 재현 불가 | `torchmetrics` 없이 scikit-learn + 직접 구현으로 대체 |
| 7 | 원본 Kaggle 데이터셋의 중복 이미지 미점검 | **Testing 394장 중 88장이 Training 과 완전히 동일** (전부 `no_tumor`) → 테스트 성능 과대평가 | 해시 기반 중복 검사와 제거 절차를 1단계에 포함 |
| 8 | 확률 캘리브레이션·해석 없음 | 임상 활용 근거 부족 | Temperature scaling·ECE·Grad-CAM 추가 |

> ### ⚠ 이 데이터셋에서 실제로 확인된 누수
>
> 이 노트북의 1단계 검사를 돌리면 다음이 나옵니다.
>
> | 항목 | 결과 |
> |---|---|
> | 파일이 **바이트 단위로 동일** (MD5) — Training ↔ Testing | 88장 |
> | **픽셀이 동일** (dHash 16×16, 재인코딩·리사이즈 포함) — Training ↔ Testing | **252장 / 394장 = 64%** |
> | 중복 252장의 클래스 | `no_tumor` **105/105 (전부)**, `meningioma` 107/115, `pituitary` 22/74, `glioma` 18/100 |
> | 중복을 뺀 '깨끗한' 테스트 세트 | 142장 — glioma 82, pituitary 52, meningioma 8, **no_tumor 0** |
> | Training 폴더 **내부** 중복 | 103장 |
> | Testing 폴더 **내부** 중복 | 82장 |
>
> 무작위로 뽑은 중복 쌍의 픽셀 평균절대오차는 **0.00** 이었습니다. 즉 JPEG 재인코딩으로 파일 바이트만 다를 뿐
> **같은 사진**입니다. 결론적으로 기존 `main.py` 가 출력하던 테스트 정확도는 **테스트의 64%가 학습에서 이미 본
> 이미지**인 상태의 수치이며, 특히 `no_tumor` 는 **단 한 장도 새로운 이미지가 아닙니다.**
>
> 그래서 이 노트북은 **Training/Testing 폴더를 합쳐 전역 중복 제거(3,264 → 2,867장) 후 다시 분할하는**
> 프로토콜을 기본값으로 씁니다. 세 가지 프로토콜을 모두 실행해 성능 차이를 비교하는 것이 이번 실습의 핵심입니다.
>
> **주의(데이터셋 한계).** 이 Kaggle 데이터셋은 환자 ID가 공개되어 있지 않습니다.
> 따라서 **환자 단위 분할이 불가능**하며, 같은 환자의 인접 슬라이스가 학습·검증에 나뉘어 들어갔을
> 가능성을 배제할 수 없습니다. 이 노트북의 성능 수치는 **교육용 상한값**으로 해석해야 하며,
> 논문에 쓸 결과라면 환자 ID가 있는 데이터셋(예: BraTS)을 사용해 GroupKFold 로 분할해야 합니다.
> 이 점 자체가 PART 5 '데이터 누수' 단원의 좋은 토론 사례입니다.

## 0. 환경 설정과 재현성

기존 실습은 Anaconda + TensorFlow 2.13(Python 3.9) 환경이었습니다. 이번에는 PyTorch 2.x 를 사용합니다.

```bash
conda create -n mlspecial python=3.11
conda activate mlspecial
# GPU (CUDA 12.x)
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124
# CPU 전용
# pip install torch torchvision
pip install timm scikit-learn matplotlib pandas pillow tqdm
```

In [ ]:
import os, sys, math, time, json, random, hashlib, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T
from PIL import Image

warnings.filterwarnings('ignore', category=UserWarning)
print('python     :', sys.version.split()[0])
print('torch      :', torch.__version__)
print('torchvision:', torchvision.__version__)
print('cuda       :', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP_DTYPE = torch.bfloat16 if DEVICE == 'cuda' and torch.cuda.is_bf16_supported() else torch.float16

# 그래프의 한글 라벨이 깨지지 않도록 폰트 설정 (Windows: Malgun Gothic / mac: AppleGothic)
import matplotlib
for _f in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR', 'DejaVu Sans']:
    if any(_f == f.name for f in matplotlib.font_manager.fontManager.ttflist):
        matplotlib.rcParams['font.family'] = _f; break
matplotlib.rcParams['axes.unicode_minus'] = False
print('matplotlib font:', matplotlib.rcParams['font.family'])

### 재현성 — 강의자료 PART 5 '재현성 체크리스트'

완전한 결정론은 GPU 연산에서 보장되지 않을 수 있으므로, **seed 를 3개 이상 바꿔가며 평균 ± 표준편차**로
보고하는 것이 원칙입니다. 아래 함수는 그 출발점입니다.

In [ ]:
def seed_everything(seed: int = 42, deterministic: bool = True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = deterministic   # 결정론적 알고리즘(속도 저하)
    torch.backends.cudnn.benchmark = not deterministic    # 입력 크기가 고정이면 True 가 빠름

def seed_worker(worker_id):
    """DataLoader 워커마다 서로 다른, 그러나 재현 가능한 시드"""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed); random.seed(worker_seed)

SEED = 42
seed_everything(SEED)
g = torch.Generator(); g.manual_seed(SEED)
print('seed fixed:', SEED)

## 1. 데이터 점검 — 클래스 분포와 중복(누수) 검사

폴더 구조는 기존과 동일합니다.

```
1.mri/
├── data/
│   ├── Training/{glioma_tumor, meningioma_tumor, no_tumor, pituitary_tumor}
│   └── Testing/ {동일한 4개 폴더}
├── main.py                              (기존 TensorFlow 코드)
└── 10주차_딥러닝분류_실습_PyTorch.ipynb  (이 노트북)
```

In [ ]:
DATA_ROOT = Path('./data')          # 노트북을 1.mri/ 안에서 실행한다고 가정
TRAIN_DIR, TEST_DIR = DATA_ROOT / 'Training', DATA_ROOT / 'Testing'
CLASSES = ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']  # 알파벳순 = 인덱스 0~3
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

assert TRAIN_DIR.exists(), f'경로를 찾을 수 없습니다: {TRAIN_DIR.resolve()}'

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}

def scan_folder(root: Path) -> pd.DataFrame:
    rows = []
    for c in CLASSES:
        for p in sorted((root / c).iterdir()):
            if p.suffix.lower() in IMG_EXT:
                rows.append({'path': str(p), 'class': c, 'label': CLASS_TO_IDX[c]})
    return pd.DataFrame(rows)

df_trainval = scan_folder(TRAIN_DIR)
df_test     = scan_folder(TEST_DIR)
print(f'Training 폴더 : {len(df_trainval):>5}장')
print(f'Testing  폴더 : {len(df_test):>5}장')
display(pd.concat([df_trainval['class'].value_counts().rename('train+val'),
                   df_test['class'].value_counts().rename('test')], axis=1).loc[CLASSES])

**불균형 확인.** `no_tumor` 클래스가 종양 클래스들의 절반 이하입니다.
정확도만 보면 소수 클래스를 놓쳐도 높게 나오므로 **macro-F1** 을 주 지표로 씁니다 (PART 5).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6.5))
for j, c in enumerate(CLASSES):
    sub = df_trainval[df_trainval['class'] == c].sample(2, random_state=SEED)
    for i, (_, r) in enumerate(sub.iterrows()):
        im = Image.open(r['path']).convert('RGB')
        axes[i, j].imshow(im); axes[i, j].axis('off')
        if i == 0:
            axes[i, j].set_title(f"{c}\n{im.size[0]}x{im.size[1]}", fontsize=10)
plt.suptitle('클래스별 샘플 이미지 (해상도가 제각각임에 주의)', fontsize=12)
plt.tight_layout(); plt.show()

sizes = Counter(Image.open(p).size for p in df_trainval['path'].sample(300, random_state=SEED))
print('이미지 크기 분포(상위 5개):', sizes.most_common(5))

### 1-2. 중복·누수 검사 (PART 5)

동일하거나 거의 같은 이미지가 Training 과 Testing 에 함께 들어 있으면 테스트 성능이 부풀려집니다.
여기서는 두 가지를 봅니다.

* **파일 중복**: 파일 바이트의 MD5 해시가 같은 경우 → 같은 파일을 복사한 것
* **픽셀 중복**: 16×16 **차이 해시(dHash)** 가 같은 경우 → JPEG 재인코딩·리사이즈로 바이트는 달라도 **같은 사진**

> dHash 는 이웃 픽셀의 밝기 대소 관계 256비트를 비교하므로, 8×8 평균 해시(aHash)보다 훨씬 판별력이 높습니다.
> (MRI 처럼 배경이 검은 영상에서 aHash 는 서로 다른 이미지도 같은 해시로 뭉뚱그립니다.)

In [ ]:
def md5_of(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

def dhash(path, size=16):
    """차이 해시: 가로 방향 이웃 픽셀의 밝기 대소 관계 size*size 비트"""
    im = Image.open(path).convert('L').resize((size + 1, size), Image.BILINEAR)
    a = np.asarray(im, dtype=np.int16)
    return np.packbits((a[:, 1:] > a[:, :-1]).flatten()).tobytes()

t0 = time.time()
for df in (df_trainval, df_test):
    df['md5'] = [md5_of(p) for p in df['path']]
    df['dh']  = [dhash(p)  for p in df['path']]
print(f'해시 계산 {time.time() - t0:.1f}s')

md5_train, dh_train = set(df_trainval['md5']), set(df_trainval['dh'])
df_test['dup_md5'] = df_test['md5'].isin(md5_train)
df_test['dup_dh']  = [h in dh_train for h in df_test['dh']]

print(f"Training 폴더 내부 중복      : MD5 {df_trainval['md5'].duplicated().sum()}장 / "
      f"dHash {df_trainval['dh'].duplicated().sum()}장")
print(f"Testing  폴더 내부 중복      : MD5 {df_test['md5'].duplicated().sum()}장 / "
      f"dHash {df_test['dh'].duplicated().sum()}장")
print(f"Training <-> Testing 파일 중복(MD5) : {df_test['dup_md5'].sum()}장 / {len(df_test)}장")
print(f"Training <-> Testing 픽셀 중복(dHash): {df_test['dup_dh'].sum()}장 / {len(df_test)}장"
      f"  ({df_test['dup_dh'].mean():.0%})")
print('\n픽셀 중복 테스트 이미지의 클래스 분포:',
      df_test[df_test['dup_dh']]['class'].value_counts().to_dict())
print("중복을 뺀 '깨끗한' 테스트 세트   :",
      df_test[~df_test['dup_dh']]['class'].value_counts().to_dict())

In [ ]:
# 정말 같은 사진인지 눈으로 확인 — dHash 는 같은데 MD5 는 다른 쌍 3개
pairs = df_test[df_test['dup_dh'] & ~df_test['dup_md5']]
if len(pairs):
    lut = {}
    for _, r in df_trainval.iterrows():
        lut.setdefault(r['dh'], r['path'])
    fig, axes = plt.subplots(2, 3, figsize=(9, 6.2))
    for k, (_, r) in enumerate(pairs.sample(min(3, len(pairs)), random_state=SEED).iterrows()):
        tr_path = lut[r['dh']]
        a = Image.open(tr_path).convert('L').resize((256, 256))
        b = Image.open(r['path']).convert('L').resize((256, 256))
        mae = np.abs(np.asarray(a, float) - np.asarray(b, float)).mean()
        axes[0, k].imshow(a, cmap='gray'); axes[0, k].axis('off')
        axes[0, k].set_title(f'Training\n{Path(tr_path).name}', fontsize=8)
        axes[1, k].imshow(b, cmap='gray'); axes[1, k].axis('off')
        axes[1, k].set_title(f'Testing  {Path(r["path"]).name}\n픽셀 MAE = {mae:.2f}', fontsize=8)
    plt.suptitle('파일은 다르지만 같은 사진 (JPEG 재인코딩)', fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print('해당하는 쌍이 없습니다.')

### 1-3. 분할 프로토콜 선택

중복이 이 정도 규모이면 "테스트에서 중복만 빼기"로는 부족합니다 — `no_tumor` 테스트가 **0장**이 되기 때문입니다.
세 가지 프로토콜을 제공하니, 같은 모델로 모두 돌려 **수치가 어떻게 달라지는지** 직접 확인하십시오.

| `PROTOCOL` | 방식 | 테스트 세트 | 의미 |
|---|---|---|---|
| `'original'` | 기존 `main.py` 와 동일 (중복 방치) | 394장 | 누수를 포함한 **낙관적** 수치 |
| `'drop_test'` | Testing 에서 픽셀 중복 제거 | 142장 (no_tumor 0장) | 4클래스 평가가 **불가능**해짐을 확인하는 용도 |
| `'repool'` **(기본·권장)** | Training+Testing 을 합쳐 전역 중복 제거 후 층화 재분할 | 2,867장을 70/15/15 로 | 누수 없는 **정직한** 평가 |

> 어떤 프로토콜을 쓰든 논문·보고서에는 **그 선택과 근거를 명시**해야 합니다.
> 출처가 제공한 분할을 그대로 쓰는 것이 항상 옳지는 않다는 것이 이 단원의 교훈입니다.

In [ ]:
from sklearn.model_selection import train_test_split

PROTOCOL = 'repool'        # 'original' | 'drop_test' | 'repool'

if PROTOCOL == 'original':                      # 기존 main.py 와 동일 (누수 방치)
    pool_tv, pool_te = df_trainval, df_test

elif PROTOCOL == 'drop_test':                   # 테스트에서 픽셀 중복만 제거
    pool_tv = df_trainval
    pool_te = df_test[~df_test['dup_dh']].reset_index(drop=True)

elif PROTOCOL == 'repool':                      # 전역 중복 제거 후 재분할 (권장)
    pool = pd.concat([df_trainval, df_test], ignore_index=True)
    before = len(pool)
    pool = pool.drop_duplicates('dh').reset_index(drop=True)   # 픽셀이 같은 사진은 1장만 남김
    print(f'전체 {before}장 -> 고유 {len(pool)}장 (중복 {before - len(pool)}장 제거)')
    pool_tv, pool_te = train_test_split(pool, test_size=0.15, random_state=SEED,
                                        stratify=pool['label'])
else:
    raise ValueError(PROTOCOL)

df_train, df_val = train_test_split(pool_tv, test_size=0.176 if PROTOCOL == 'repool' else 0.2,
                                    random_state=SEED, stratify=pool_tv['label'])
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = pool_te.reset_index(drop=True)

print(f"\n[{PROTOCOL}]  train {len(df_train)} / val {len(df_val)} / test {len(df_test)}")
print('train:', df_train['class'].value_counts().to_dict())
print('val  :', df_val['class'].value_counts().to_dict())
print('test :', df_test['class'].value_counts().to_dict())

# 분할 인덱스를 파일로 남겨 재현 가능하게 (PART 5: 재현성 체크리스트)
for name, d in [('train', df_train), ('val', df_val), ('test', df_test)]:
    d[['path', 'class', 'label']].to_csv(f'split_{PROTOCOL}_{name}_seed{SEED}.csv',
                                         index=False, encoding='utf-8-sig')
print('분할 인덱스를 split_*.csv 로 저장했습니다.')

**기존 코드와의 차이 (검증 비율).** 기존에는 `test_size=1-0.6` 즉 40%를 검증에 썼습니다.
검증 세트가 클수록 모델 선택은 안정되지만 학습 데이터가 줄어듭니다. 데이터가 2,870장뿐이므로
20% (574장)로도 충분히 안정적이며, 더 엄밀하게 하려면 **5-겹 교차검증**을 쓰는 것이 좋습니다
(마지막 과제 참조).

## 2. Dataset · DataLoader · 데이터 증강 (PART 3)

기존 코드는 전체 이미지를 NumPy 배열로 메모리에 올렸습니다(≈2.2 GB).
PyTorch 의 `Dataset` 은 **필요한 순간에 한 장씩** 읽으므로 메모리를 거의 쓰지 않고,
`DataLoader` 가 여러 워커로 디코딩·증강을 병렬 처리합니다.

In [ ]:
IMG_SIZE = 224                      # ImageNet 사전학습 모델의 표준 입력
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    T.RandomCrop(IMG_SIZE),
    T.RandomHorizontalFlip(),                 # 뇌 MRI 축상면은 좌우 대칭 → 허용
    T.RandomRotation(10),                     # 촬영 각도 변동 모사
    T.RandAugment(num_ops=2, magnitude=7),    # 자동 증강 정책 (강도는 실험으로 조정)
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=0.25, scale=(0.02, 0.1)),
])

eval_tf = T.Compose([                          # 검증·테스트에는 증강을 쓰지 않는다
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class MRIDataset(Dataset):
    """경로 목록(DataFrame)을 받아 이미지를 지연 로딩하는 데이터셋"""
    def __init__(self, df: pd.DataFrame, transform=None):
        self.paths = df['path'].tolist()
        self.labels = df['label'].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')   # 흑백 MRI 지만 3채널로 (사전학습 모델 호환)
        if self.transform is not None:
            img = self.transform(img)
        return img, self.labels[i]

BATCH_SIZE  = 32
NUM_WORKERS = min(4, os.cpu_count() or 1)

def make_loader(df, transform, shuffle=False, sampler=None, bs=BATCH_SIZE):
    return DataLoader(MRIDataset(df, transform), batch_size=bs,
                      shuffle=(shuffle and sampler is None), sampler=sampler,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE == 'cuda'),
                      persistent_workers=NUM_WORKERS > 0, drop_last=False,
                      worker_init_fn=seed_worker, generator=g)

train_loader = make_loader(df_train, train_tf, shuffle=True)
val_loader   = make_loader(df_val,   eval_tf)
test_loader  = make_loader(df_test,  eval_tf)

xb, yb = next(iter(train_loader))
print('배치 모양:', xb.shape, xb.dtype, '| 라벨:', yb[:8].tolist())
print('정규화 후 값 범위: [%.2f, %.2f]' % (xb.min(), xb.max()))

In [ ]:
# 증강 결과 눈으로 확인하기 (같은 이미지를 6번 통과시킴)
inv = T.Normalize([-m/s for m, s in zip(IMAGENET_MEAN, IMAGENET_STD)], [1/s for s in IMAGENET_STD])
sample_path = df_train['path'].iloc[0]
fig, axes = plt.subplots(1, 6, figsize=(15, 2.8))
axes[0].imshow(Image.open(sample_path).convert('RGB')); axes[0].set_title('원본'); axes[0].axis('off')
for k in range(1, 6):
    t = inv(train_tf(Image.open(sample_path).convert('RGB'))).clamp(0, 1).permute(1, 2, 0).numpy()
    axes[k].imshow(t); axes[k].set_title(f'증강 {k}'); axes[k].axis('off')
plt.tight_layout(); plt.show()

> **의료 영상에서 증강을 고를 때의 원칙 (토론 주제).**
> 좌우 반전은 뇌 축상면에서는 대체로 안전하지만, **좌우 비대칭 자체가 소견인 경우**(예: 편측 병변의 위치 보고)에는
> 라벨의 의미를 바꿀 수 있습니다. 밝기·대비 변형도 창(window) 설정이 진단 단서인 모달리티에서는 신중해야 합니다.
> 증강은 "데이터가 실제로 가질 수 있는 변동"만 모사해야 합니다.

### 2-3. Mixup · CutMix (PART 3)

강의자료의 `v2.MixUp` / `v2.CutMix` 를 직접 구현한다. 두 기법 모두 **두 표본을 섞고 라벨도 같은 비율로 섞어**
결정 경계를 부드럽게 만든다. 섞인 라벨은 더 이상 정수가 아니라 **(N, K) 확률 벡터(소프트 라벨)** 이며,
`nn.CrossEntropyLoss` 는 PyTorch 1.10 부터 이 형태를 그대로 받는다.

* **Mixup**: `x = λ·x₁ + (1−λ)·x₂`, `y = λ·y₁ + (1−λ)·y₂`
* **CutMix**: 사각 영역을 잘라 붙이고, **실제로 잘린 면적 비**로 `λ` 를 다시 계산해 라벨을 섞는다

> Mixup/CutMix 를 쓰면 **학습 손실이 높게 보이는 것이 정상**이다. 검증 지표로 판단해야 한다.


In [ ]:
class MixupCutmix:
    """배치 단위로 Mixup(Zhang+,2018) 또는 CutMix(Yun+,2019) 를 확률적으로 적용한다.

    반환: (섞인 이미지 x, 소프트 라벨 y)  — y 는 (N, K) 확률 벡터로 각 행의 합이 1 이다.
    """
    def __init__(self, num_classes=NUM_CLASSES, alpha=0.2, prob=0.5, switch_prob=0.5):
        self.K, self.alpha = num_classes, alpha
        self.prob, self.switch = prob, switch_prob      # prob: 적용 확률, switch: CutMix 선택 확률

    def _onehot(self, y):
        return F.one_hot(y, self.K).float()

    def __call__(self, x, y):
        y1 = self._onehot(y)
        if self.alpha <= 0 or random.random() > self.prob:
            return x, y1                                     # 이번 배치는 섞지 않는다
        lam = float(np.random.beta(self.alpha, self.alpha))   # 혼합 비율 λ ~ Beta(α, α)
        idx = torch.randperm(x.size(0), device=x.device)      # 짝이 될 상대 표본의 순열
        if random.random() < self.switch:                     # ---------- CutMix ----------
            H, W = x.shape[-2:]
            rh, rw = int(H * math.sqrt(1 - lam)), int(W * math.sqrt(1 - lam))   # 잘라낼 창 크기
            cy, cx = np.random.randint(H), np.random.randint(W)                 # 창의 중심
            ya, yb = max(cy - rh // 2, 0), min(cy + rh // 2, H)
            xa, xb = max(cx - rw // 2, 0), min(cx + rw // 2, W)
            x = x.clone()
            x[:, :, ya:yb, xa:xb] = x[idx][:, :, ya:yb, xa:xb]
            lam = 1.0 - (yb - ya) * (xb - xa) / (H * W)       # 실제 잘린 면적 비로 λ 보정
        else:                                                 # ---------- Mixup -----------
            x = lam * x + (1 - lam) * x[idx]
        return x, lam * y1 + (1 - lam) * y1[idx]

mix_fn = MixupCutmix(alpha=0.2, prob=0.5)          # 실제 학습에 쓸 설정
_demo  = MixupCutmix(alpha=0.4, prob=1.0)          # 확인용 — 항상 섞이도록 prob=1.0
_xb = torch.randn(8, 3, 32, 32); _yb = torch.randint(0, NUM_CLASSES, (8,))
_x, _y = _demo(_xb, _yb)
print('섞은 뒤 x:', tuple(_x.shape), '| y:', tuple(_y.shape))
print('소프트 라벨 각 행의 합 (1 이어야 함):', [round(v, 4) for v in _y.sum(1).tolist()[:4]])
print('소프트 라벨 예시:', [round(v, 3) for v in _y[0].tolist()])
assert torch.allclose(_y.sum(1), torch.ones(8)), '소프트 라벨의 행 합이 1 이 아닙니다'
assert not torch.equal(_x, _xb), 'prob=1.0 인데 이미지가 섞이지 않았습니다'

# 사용법:  train_model(SimpleCNN(), train_loader, val_loader, epochs=EPOCHS, mixup=mix_fn)


## 3. 기준선 모델 — SimpleCNN (PART 2)

강의자료의 `SimpleCNN` 을 4클래스·224×224 입력에 맞게 확장했습니다.
`Conv → BN → ReLU → MaxPool` 블록을 4번 쌓고, **전역 평균 풀링(GAP)** 으로 분류기를 만듭니다
(Flatten + 큰 FC 대신 GAP 를 쓰면 파라미터가 크게 줄어듭니다).

In [ ]:
def conv_block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),   # BN 이 뒤따르므로 bias 불필요
        nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1, bias=False),
        nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.MaxPool2d(2),                                     # 공간 크기 1/2
    )

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, widths=(32, 64, 128, 256), in_ch=3):
        super().__init__()
        blocks, c_prev = [], in_ch
        for w in widths:
            blocks.append(conv_block(c_prev, w)); c_prev = w
        self.features = nn.Sequential(*blocks)               # 224 -> 14
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),           # (N, C, H, W) -> (N, C)
            nn.Dropout(0.3), nn.Linear(c_prev, num_classes)) # 로짓 출력
    def forward(self, x):
        return self.head(self.features(x))

def count_params(m):
    total = sum(p.numel() for p in m.parameters())
    train = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, train

m = SimpleCNN()
with torch.no_grad():
    out = m(torch.randn(2, 3, IMG_SIZE, IMG_SIZE))
print('출력 모양:', out.shape)
print('파라미터: 전체 %s / 학습 %s' % tuple(f'{v:,}' for v in count_params(m)))

# 층별 텐서 모양 추적 (강의자료 '층을 지날 때 텐서 모양의 변화' 슬라이드와 동일한 확인)
h = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
for i, blk in enumerate(m.features):
    h = blk(h); print(f'  block{i+1}: {tuple(h.shape)}')

### 3-2. 출력 크기·파라미터 수·수용영역을 손으로 계산하기 (PART 2)

라이브러리가 알려주기 전에 **직접 계산할 수 있어야** 설계를 할 수 있다. 강의자료의 세 공식을 함수로 옮기고,
PyTorch 가 실제로 만든 텐서·파라미터와 일치하는지 `assert` 로 확인한다.

* 출력 크기: `⌊(n + 2p − d(k−1) − 1)/s⌋ + 1`
* 합성곱 파라미터 수: `k·k·C_in·C_out (+ C_out)`  — BatchNorm 은 채널당 `γ, β` 2개
* 수용영역 재귀: `r_L = r_(L−1) + (k_L − 1)·Π_(i<L) s_i`


In [ ]:
def conv_out_size(n, k, s=1, p=0, d=1):
    """합성곱·풀링 한 층을 지난 뒤의 한 변 길이. 정수를 반환한다."""
    return (n + 2 * p - d * (k - 1) - 1) // s + 1

def conv_params(k, c_in, c_out, bias=True):
    """k×k 합성곱 한 층의 파라미터 수. bias=False 면 편향을 세지 않는다."""
    return k * k * c_in * c_out + (c_out if bias else 0)

def receptive_field(layers):
    """수용영역 재귀. layers 는 [(k, s), ...] 형태의 (커널, 스트라이드) 목록.

    반환: 마지막 층 출력 한 픽셀이 입력에서 보는 정사각 영역의 한 변 길이(정수).
    """
    r, jump = 1, 1                     # r: 현재 수용영역, jump: 누적 스트라이드 Π s_i
    for k, s in layers:
        r += (k - 1) * jump
        jump *= s
    return r

# (1) 출력 크기 — SimpleCNN 의 블록을 손으로 따라가 본다 (3x3 'same' 두 번 + MaxPool2d(2))
n = IMG_SIZE
for i, w in enumerate((32, 64, 128, 256), 1):
    n = conv_out_size(conv_out_size(n, 3, s=1, p=1), 3, s=1, p=1)
    n = conv_out_size(n, 2, s=2)
    print(f'  block{i}: {n}x{n}')
assert n == 14, f'마지막 특징 맵은 14x14 여야 합니다 (계산값 {n})'

# (2) 파라미터 수 — 손 계산과 PyTorch 를 맞춰본다
hand, c_prev = 0, 3
for w in (32, 64, 128, 256):
    hand += conv_params(3, c_prev, w, bias=False) + 2 * w   # conv1 + BN1 (gamma, beta)
    hand += conv_params(3, w, w, bias=False) + 2 * w        # conv2 + BN2
    c_prev = w
hand += c_prev * NUM_CLASSES + NUM_CLASSES                  # GAP 뒤의 Linear
torch_total = count_params(SimpleCNN())[0]
print('\n파라미터 수  손 계산 %s / PyTorch %s' % (f'{hand:,}', f'{torch_total:,}'))
assert hand == torch_total, '손 계산과 PyTorch 가 다릅니다 — 어디를 빠뜨렸는지 찾아보십시오'

# (3) 수용영역 — 3x3 두 개가 5x5 하나와 같다는 PART 4 의 주장 확인
print('3x3 합성곱 두 개의 수용영역 :', receptive_field([(3, 1), (3, 1)]), '(5x5 하나와 동일)')
print('SimpleCNN 전체의 수용영역   :', receptive_field([(3, 1), (3, 1), (2, 2)] * 4), '픽셀')
print('ResNet-18 stem 까지의 수용영역:', receptive_field([(7, 2), (3, 2)]), '픽셀')


## 4. 학습 루프 (PART 3: AdamW · 워밍업+코사인 · 라벨 스무딩 · AMP)

Keras 의 `model.fit` 에 해당하는 부분을 직접 작성합니다. 강의자료에서 다룬 기법을 모두 반영했습니다.

* `nn.CrossEntropyLoss(label_smoothing=0.1)` — 과신 완화
* `AdamW` + **워밍업 → 코사인 감쇠** 스케줄
* `torch.autocast` 혼합 정밀도 (bf16 / fp16 + `GradScaler`)
* 검증 **macro-F1 최대** 기준 체크포인트 (기존 코드의 `mode='max'` 버그를 올바르게 구현)

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report, roc_auc_score)
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

@torch.no_grad()
def predict(model, loader, device=DEVICE):
    """로짓·확률·예측·정답을 한 번에 반환"""
    model.eval()
    logits_all, y_all = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        with torch.autocast(device_type=device, dtype=AMP_DTYPE, enabled=(device == 'cuda')):
            logits = model(x)
        logits_all.append(logits.float().cpu()); y_all.append(y)
    logits = torch.cat(logits_all); y = torch.cat(y_all)
    probs = logits.softmax(1)
    return logits, probs, probs.argmax(1), y

def metrics_from(probs, pred, y):
    # float() 로 감싼다: sklearn 이 돌려주는 numpy 스칼라를 그대로 체크포인트에 넣으면
    # torch>=2.6 의 torch.load(weights_only=True) 가 UnpicklingError 로 거부한다.
    out = {'acc': float(accuracy_score(y, pred)),
           'macro_f1': float(f1_score(y, pred, average='macro'))}
    try:
        out['auroc'] = float(roc_auc_score(y, probs, multi_class='ovr', average='macro'))
    except ValueError:
        out['auroc'] = float('nan')          # 검증 배치에 없는 클래스가 있으면 계산 불가
    return out

def make_scheduler(optimizer, epochs, warmup=3, steps_per_epoch=1):
    warmup = max(1, min(warmup, epochs - 1)) if epochs > 1 else 1   # 워밍업 > 총 에폭 방지
    warm = LinearLR(optimizer, start_factor=0.01, total_iters=warmup * steps_per_epoch)
    cos  = CosineAnnealingLR(optimizer, T_max=max(1, (epochs - warmup) * steps_per_epoch))
    return SequentialLR(optimizer, [warm, cos], milestones=[warmup * steps_per_epoch])

def train_model(model, train_loader, val_loader, epochs=20, lr=1e-3, weight_decay=0.05,
                label_smoothing=0.1, class_weight=None, warmup=3, ckpt='best.pt',
                mixup=None, log_every=1, device=DEVICE):
    model = model.to(device, memory_format=torch.channels_last)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing,
                                    weight=None if class_weight is None else class_weight.to(device))
    # 가중치 감쇠는 bias·정규화 층 파라미터에는 적용하지 않는다 (PART 3)
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        (no_decay if p.ndim <= 1 else decay).append(p)
    optimizer = torch.optim.AdamW([{'params': decay,    'weight_decay': weight_decay},
                                   {'params': no_decay, 'weight_decay': 0.0}], lr=lr)
    sched  = make_scheduler(optimizer, epochs, warmup)
    scaler = torch.amp.GradScaler(device, enabled=(device == 'cuda' and AMP_DTYPE is torch.float16))

    hist, best = [], {'macro_f1': -1}
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); run_loss, n_seen = 0.0, 0
        for x, y in train_loader:
            x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)
            if mixup is not None:
                x, y = mixup(x, y)                       # y 가 소프트 라벨 (N, K) 로 바뀜
            with torch.autocast(device_type=device, dtype=AMP_DTYPE, enabled=(device == 'cuda')):
                loss = criterion(model(x), y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # 손실 발산 방지
            scaler.step(optimizer); scaler.update()
            run_loss += loss.item() * x.size(0); n_seen += x.size(0)
        sched.step()

        _, probs, pred, yv = predict(model, val_loader, device)
        mtr = metrics_from(probs.numpy(), pred.numpy(), yv.numpy())
        rec = {'epoch': ep, 'train_loss': run_loss / n_seen,
               'lr': optimizer.param_groups[0]['lr'], **mtr, 'sec': time.time() - t0}
        hist.append(rec)
        if mtr['macro_f1'] > best['macro_f1']:            # 최대화 기준으로 저장
            best = {**rec}
            torch.save({'model': model.state_dict(), 'epoch': ep, **mtr}, ckpt)
        if ep % log_every == 0 or ep == 1:
            print(f"ep {ep:>3}/{epochs} loss {rec['train_loss']:.4f} "
                  f"val_acc {mtr['acc']:.4f} val_F1 {mtr['macro_f1']:.4f} "
                  f"AUROC {mtr['auroc']:.4f} lr {rec['lr']:.2e} ({rec['sec']:.1f}s)")
    model.load_state_dict(torch.load(ckpt, map_location=device)['model'])   # 최고 성능 가중치 복원
    print(f"\n[best] epoch {best['epoch']} · val macro-F1 {best['macro_f1']:.4f}")
    return model, pd.DataFrame(hist), best

In [ ]:
EPOCHS = 20            # 강의 시간에 맞춰 조정 (GPU 없으면 3~5 로 줄이세요)
seed_everything(SEED)
baseline = SimpleCNN()
baseline, hist_base, best_base = train_model(
    baseline, train_loader, val_loader, epochs=EPOCHS, lr=1e-3, ckpt='ckpt_simplecnn.pt')

In [ ]:
def plot_history(hists: dict):
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
    for name, h in hists.items():
        axes[0].plot(h['epoch'], h['train_loss'], label=name)
        axes[1].plot(h['epoch'], h['macro_f1'],  label=name)
        axes[2].plot(h['epoch'], h['lr'],        label=name)
    for ax, t in zip(axes, ['학습 손실', '검증 macro-F1', '학습률']):
        ax.set_title(t); ax.set_xlabel('epoch'); ax.grid(alpha=.3); ax.legend(fontsize=9)
    axes[2].set_yscale('log')
    plt.tight_layout(); plt.show()

plot_history({'SimpleCNN': hist_base})

## 5. 평가 — 정확도만으로는 부족하다 (PART 2·5)

혼동행렬, 클래스별 정밀도/재현율/F1, macro-AUROC 를 함께 봅니다.

In [ ]:
def plot_confusion(y, pred, classes=CLASSES, normalize=True, title='혼동행렬'):
    cm = confusion_matrix(y, pred, labels=range(len(classes)))
    cmn = cm.astype(float) / cm.sum(1, keepdims=True).clip(min=1) if normalize else cm
    fig, ax = plt.subplots(figsize=(5.4, 4.6))
    im = ax.imshow(cmn, cmap='Greens', vmin=0, vmax=cmn.max())
    ax.set_xticks(range(len(classes))); ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=35, ha='right', fontsize=9)
    ax.set_yticklabels(classes, fontsize=9)
    ax.set_xlabel('예측'); ax.set_ylabel('실제'); ax.set_title(title)
    for i in range(len(classes)):
        for j in range(len(classes)):
            txt = f'{cmn[i, j]:.2f}\n({cm[i, j]})' if normalize else f'{cm[i, j]}'
            ax.text(j, i, txt, ha='center', va='center', fontsize=8,
                    color='white' if cmn[i, j] > cmn.max() * .6 else 'black')
    fig.colorbar(im, fraction=.046); plt.tight_layout(); plt.show()

def evaluate_report(model, loader, name='model'):
    logits, probs, pred, y = predict(model, loader)
    mtr = metrics_from(probs.numpy(), pred.numpy(), y.numpy())
    print(f'== {name} ==  acc {mtr["acc"]:.4f} | macro-F1 {mtr["macro_f1"]:.4f} | AUROC {mtr["auroc"]:.4f}')
    print(classification_report(y, pred, target_names=CLASSES, digits=3, zero_division=0))
    plot_confusion(y, pred, title=f'{name} — 테스트 혼동행렬')
    return {'name': name, **mtr, 'logits': logits, 'probs': probs, 'pred': pred, 'y': y}

res_base = evaluate_report(baseline, test_loader, 'SimpleCNN (기준선)')

## 6. ResNet-18 직접 구현 (PART 4: 잔차 연결)

강의자료의 `BasicBlock` 을 그대로 쓰고 스테이지를 쌓아 ResNet-18 을 만듭니다.
`y = F(x) + x` 의 항등 경로가 기울기를 그대로 흘려보내 깊은 신경망의 학습을 가능하게 합니다.

In [ ]:
class BasicBlock(nn.Module):
    """ResNet 기본 블록. residual=False 로 두면 잔차 연결이 없는 plain 블록이 된다."""
    expansion = 1
    def __init__(self, c_in, c_out, stride=1, residual=True):
        super().__init__()
        self.residual = residual
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(c_out)
        self.shortcut = nn.Identity()
        if stride != 1 or c_in != c_out:                    # 모양이 다르면 1x1 투영
            self.shortcut = nn.Sequential(
                nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if not self.residual:
            return F.relu(out)                              # plain: x 를 더하지 않음
        return F.relu(out + self.shortcut(x))               # F(x) + x

class ResNet18(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, widths=(64, 128, 256, 512), blocks=(2, 2, 2, 2),
                 residual=True):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, widths[0], 7, 2, 3, bias=False),   # 224 -> 112
            nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2, 1))                          # 112 -> 56
        layers, c_prev = [], widths[0]
        for i, (w, n) in enumerate(zip(widths, blocks)):
            for b in range(n):
                layers.append(BasicBlock(c_prev, w, stride=2 if (b == 0 and i > 0) else 1,
                                         residual=residual))
                c_prev = w
        self.stages = nn.Sequential(*layers)                # 56 -> 7
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Linear(c_prev, num_classes))
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')  # He 초기화
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
        for m in self.modules():                            # zero-init residual: 초기엔 항등 사상
            if isinstance(m, BasicBlock) and m.residual:
                nn.init.zeros_(m.bn2.weight)
    def forward(self, x):
        return self.head(self.stages(self.stem(x)))

r18 = ResNet18()
print('출력 모양:', r18(torch.randn(2, 3, IMG_SIZE, IMG_SIZE)).shape)
print('파라미터: 전체 %s / 학습 %s' % tuple(f'{v:,}' for v in count_params(r18)))

ref = torchvision.models.resnet18(num_classes=NUM_CLASSES)
print('torchvision resnet18 파라미터: %s (직접 구현과 거의 동일해야 함)' % f'{count_params(ref)[0]:,}')

In [ ]:
seed_everything(SEED)
resnet_scratch, hist_r18, best_r18 = train_model(
    ResNet18(), train_loader, val_loader, epochs=EPOCHS, lr=1e-3, ckpt='ckpt_resnet18_scratch.pt')
res_r18 = evaluate_report(resnet_scratch, test_loader, 'ResNet-18 (scratch)')

### 6-2. 잔차 연결의 효과 확인 (선택 실험)

`BasicBlock.forward` 의 `+ self.shortcut(x)` 를 제거한 **plain** 버전을 같은 조건으로 학습해
학습 곡선을 비교하면, 강의자료에서 말한 '퇴화(degradation) 문제'를 직접 관찰할 수 있습니다.
층이 얕으면 차이가 작으므로 `blocks=(3,4,6,3)` 이상으로 깊게 만들어야 효과가 드러납니다.

In [ ]:
# 깊은(34층) 설정에서 잔차 연결 유무만 바꾼 두 모델
deep_res   = ResNet18(blocks=(3, 4, 6, 3), residual=True)
deep_plain = ResNet18(blocks=(3, 4, 6, 3), residual=False)
print('ResNet-34  파라미터: %s' % f'{count_params(deep_res)[0]:,}')
print('Plain-34   파라미터: %s  (shortcut 의 1x1 투영만큼만 차이)' % f'{count_params(deep_plain)[0]:,}')
print('출력 모양:', deep_plain(torch.randn(2, 3, IMG_SIZE, IMG_SIZE)).shape)

# 시간이 허락하면 아래 세 줄을 실행해 학습 곡선을 비교하십시오 (각각 수십 분 소요).
# _, hist_plain, _ = train_model(deep_plain, train_loader, val_loader, epochs=EPOCHS, ckpt='ckpt_plain34.pt')
# _, hist_res34, _ = train_model(deep_res,   train_loader, val_loader, epochs=EPOCHS, ckpt='ckpt_res34.pt')
# plot_history({'Plain-34': hist_plain, 'ResNet-34': hist_res34})

## 7. 전이학습 (PART 5)

소규모 의료 영상에서는 **처음부터 학습하는 것보다 사전학습 가중치를 미세조정하는 쪽이 거의 항상 낫습니다.**

전략 두 가지를 모두 실행해 비교합니다.
1. **특징 추출(linear probe)**: 백본을 고정하고 분류기 헤드만 학습 → 빠른 기준선
2. **미세조정(fine-tuning)**: 백본을 낮은 학습률로 함께 갱신 → 보통 더 높은 성능

> 사전학습 가중치는 첫 실행 시 인터넷에서 내려받습니다(수십~수백 MB). 오프라인 환경이라면
> 미리 받아 `~/.cache/torch/hub/checkpoints/` 에 두십시오.

In [ ]:
def build_pretrained(name='resnet50', num_classes=NUM_CLASSES, freeze_backbone=False):
    """torchvision 사전학습 모델을 불러와 헤드를 교체"""
    if name == 'resnet50':
        w = torchvision.models.ResNet50_Weights.IMAGENET1K_V2      # 현대 레시피로 재학습된 가중치 (top-1 80.9%)
        m = torchvision.models.resnet50(weights=w)
        in_f = m.fc.in_features; m.fc = nn.Linear(in_f, num_classes); head_names = ('fc',)
    elif name == 'convnext_tiny':
        w = torchvision.models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
        m = torchvision.models.convnext_tiny(weights=w)
        in_f = m.classifier[2].in_features
        m.classifier[2] = nn.Linear(in_f, num_classes); head_names = ('classifier',)
    else:
        raise ValueError(name)
    if freeze_backbone:
        for n_, p in m.named_parameters():
            p.requires_grad = n_.startswith(head_names)
    return m, head_names

def param_groups_llrd(model, head_names, lr_backbone=1e-4, lr_head=1e-3):
    """층별 학습률(discriminative lr): 백본은 작게, 새 헤드는 크게"""
    backbone = [p for n, p in model.named_parameters()
                if p.requires_grad and not n.startswith(head_names)]
    head     = [p for n, p in model.named_parameters()
                if p.requires_grad and n.startswith(head_names)]
    return [{'params': backbone, 'lr': lr_backbone}, {'params': head, 'lr': lr_head}]

try:
    tl_model, head_names = build_pretrained('resnet50', freeze_backbone=True)
    print('사전학습 가중치 로드 성공')
    print('학습 파라미터: %s / 전체 %s' % tuple(f'{v:,}' for v in count_params(tl_model)[::-1]))
    PRETRAINED_OK = True
except Exception as e:
    print('사전학습 가중치를 받지 못했습니다 (오프라인?):', type(e).__name__, str(e)[:100])
    print('-> weights=None 으로 구조만 사용합니다. 인터넷 연결 후 다시 실행하세요.')
    tl_model = torchvision.models.resnet50(weights=None, num_classes=NUM_CLASSES)
    head_names = ('fc',); PRETRAINED_OK = False

In [ ]:
# --- 1단계: 헤드만 학습 (백본 고정) ---
seed_everything(SEED)
tl_model, hist_probe, _ = train_model(
    tl_model, train_loader, val_loader, epochs=max(3, EPOCHS // 4),
    lr=1e-3, weight_decay=0.01, warmup=1, ckpt='ckpt_r50_probe.pt')

In [ ]:
# --- 2단계: 백본 해동 후 층별 학습률로 전체 미세조정 ---
for p in tl_model.parameters():
    p.requires_grad = True

def train_finetune(model, head_names, epochs=10, lr_backbone=1e-4, lr_head=1e-3,
                   weight_decay=0.05, ckpt='ckpt_r50_ft.pt', device=DEVICE):
    """train_model 과 동일하지만 옵티마이저를 층별 학습률로 구성"""
    model = model.to(device, memory_format=torch.channels_last)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(param_groups_llrd(model, head_names, lr_backbone, lr_head),
                                  weight_decay=weight_decay)
    sched  = make_scheduler(optimizer, epochs, warmup=1)
    scaler = torch.amp.GradScaler(device, enabled=(device == 'cuda' and AMP_DTYPE is torch.float16))
    hist, best = [], {'macro_f1': -1}
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); run, n_seen = 0.0, 0
        for x, y in train_loader:
            x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)
            with torch.autocast(device_type=device, dtype=AMP_DTYPE, enabled=(device == 'cuda')):
                loss = criterion(model(x), y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            run += loss.item() * x.size(0); n_seen += x.size(0)
        sched.step()
        _, probs, pred, yv = predict(model, val_loader, device)
        mtr = metrics_from(probs.numpy(), pred.numpy(), yv.numpy())
        rec = {'epoch': ep, 'train_loss': run / n_seen,
               'lr': optimizer.param_groups[0]['lr'], **mtr, 'sec': time.time() - t0}
        hist.append(rec)
        if mtr['macro_f1'] > best['macro_f1']:
            best = {**rec}; torch.save({'model': model.state_dict(), **mtr}, ckpt)
        print(f"ep {ep:>3}/{epochs} loss {rec['train_loss']:.4f} "
              f"val_F1 {mtr['macro_f1']:.4f} AUROC {mtr['auroc']:.4f} ({rec['sec']:.1f}s)")
    model.load_state_dict(torch.load(ckpt, map_location=device)['model'])
    return model, pd.DataFrame(hist), best

seed_everything(SEED)
tl_model, hist_ft, best_ft = train_finetune(tl_model, head_names, epochs=max(3, EPOCHS // 2))
res_tl = evaluate_report(tl_model, test_loader, 'ResNet-50 (전이학습)')

### 7-2. ConvNeXt-T 로 바꿔보기 (PART 4)

`timm` 을 쓰면 ImageNet-22k 사전학습 가중치와 전처리 설정을 한 번에 가져올 수 있습니다.

In [ ]:
# import timm
# model_cnx = timm.create_model('convnext_tiny.fb_in22k_ft_in1k', pretrained=True,
#                               num_classes=NUM_CLASSES, drop_path_rate=0.1)
# cfg = timm.data.resolve_data_config({}, model=model_cnx)
# print('timm 권장 전처리:', cfg)          # 입력 크기·정규화 값을 사전학습과 맞추는 것이 중요
# seed_everything(SEED)
# model_cnx, hist_cnx, _ = train_finetune(model_cnx, ('head',), epochs=max(3, EPOCHS // 2),
#                                         lr_backbone=5e-5, lr_head=5e-4, ckpt='ckpt_convnext.pt')
# res_cnx = evaluate_report(model_cnx, test_loader, 'ConvNeXt-T (전이학습)')
print('timm 설치 후 위 주석을 해제하세요:  pip install timm')

## 8. 클래스 불균형 다루기 (PART 5)

`no_tumor` 가 소수 클래스입니다. 세 가지 접근을 제공합니다.

1. **손실 가중치** — 역빈도 가중치를 `CrossEntropyLoss(weight=...)` 에
2. **균형 샘플링** — `WeightedRandomSampler` 로 배치를 균형 있게 구성
3. **Focal loss** — 쉬운 예제의 손실을 낮춰 어려운(소수) 예제에 집중

In [ ]:
counts = torch.tensor([ (df_train['label'] == i).sum() for i in range(NUM_CLASSES) ], dtype=torch.float)
print('클래스별 학습 표본 수:', dict(zip(CLASSES, counts.int().tolist())))

# (1) 역빈도 손실 가중치 — 평균이 1이 되도록 정규화
class_weight = (1.0 / counts); class_weight = class_weight / class_weight.sum() * NUM_CLASSES
print('손실 가중치:', dict(zip(CLASSES, [round(v, 3) for v in class_weight.tolist()])))

# (2) 균형 샘플링용 샘플러
sample_w = (1.0 / counts)[torch.tensor(df_train['label'].values)]
balanced_sampler = WeightedRandomSampler(sample_w.double(), num_samples=len(sample_w), replacement=True)
balanced_loader  = make_loader(df_train, train_tf, sampler=balanced_sampler)
yb = torch.cat([y for _, y in [next(iter(balanced_loader)) for _ in range(5)]])
print('균형 샘플링 배치의 클래스 분포:', Counter(yb.tolist()))

# (3) Focal loss
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__(); self.gamma, self.weight, self.ls = gamma, weight, label_smoothing
    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, weight=self.weight,
                             label_smoothing=self.ls, reduction='none')
        pt = torch.exp(-ce)                       # 정답 클래스 확률
        return ((1 - pt) ** self.gamma * ce).mean()
print('FocalLoss 예시 손실:', float(FocalLoss()(torch.randn(8, NUM_CLASSES), torch.randint(0, NUM_CLASSES, (8,)))))

In [ ]:
# 손실 가중치를 적용해 기준선을 다시 학습하고 소수 클래스 F1 의 변화를 확인
seed_everything(SEED)
balanced_model, hist_bal, _ = train_model(
    SimpleCNN(), balanced_loader, val_loader, epochs=EPOCHS,
    lr=1e-3, class_weight=class_weight, ckpt='ckpt_simplecnn_balanced.pt')
res_bal = evaluate_report(balanced_model, test_loader, 'SimpleCNN + 불균형 처리')

## 9. 확률 캘리브레이션 (PART 5)

정확도가 같아도 **확률이 과신되어 있으면** 임상 의사결정에 쓸 수 없습니다.
검증 세트로 온도 `T` 를 학습하고(순위는 바뀌지 않음), 테스트 세트에서 ECE 를 보고합니다.

In [ ]:
def fit_temperature(logits, y, max_iter=200):
    """검증 세트 로짓으로 온도 T 를 학습 (NLL 최소화). 정확도(순위)는 바뀌지 않는다."""
    logits, y = logits.detach().float(), y.detach()
    logT = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=0.05, max_iter=max_iter, line_search_fn='strong_wolfe')
    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(logits / logT.exp().clamp(min=1e-2), y)
        loss.backward(); return loss
    opt.step(closure)
    with torch.no_grad():
        t = float(logT.exp().clamp(0.05, 20.0))
    return t if np.isfinite(t) else 1.0          # 발산하면 보정하지 않음(T=1)

def expected_calibration_error(probs, y, n_bins=15):
    conf, pred = probs.max(1)
    acc = (pred == y).float()
    bins, ece = torch.linspace(0, 1, n_bins + 1), 0.0
    stats = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            a, c, w = acc[m].mean(), conf[m].mean(), m.float().mean()
            ece += w * (a - c).abs(); stats.append((float(lo), float(a), float(c), int(m.sum())))
    return float(ece), stats

# 검증 세트로 T 학습 → 테스트 세트에 적용
val_logits, _, _, val_y = predict(tl_model, val_loader)
T = fit_temperature(val_logits, val_y)
test_logits, test_probs, _, test_y = predict(tl_model, test_loader)
probs_cal = (test_logits / T).softmax(1)

ece_before, st_b = expected_calibration_error(test_probs, test_y)
ece_after,  st_a = expected_calibration_error(probs_cal,  test_y)
nll_b = float(F.cross_entropy(test_logits, test_y))
nll_a = float(F.cross_entropy(test_logits / T, test_y))
print(f'학습된 온도 T = {T:.3f}')
print(f'ECE  보정 전 {ece_before:.4f} -> 보정 후 {ece_after:.4f}')
print(f'NLL  보정 전 {nll_b:.4f} -> 보정 후 {nll_a:.4f}')
print(f'정확도(불변): {(test_probs.argmax(1)==test_y).float().mean():.4f} / '
      f'{(probs_cal.argmax(1)==test_y).float().mean():.4f}')

In [ ]:
# 신뢰도 다이어그램 (reliability diagram) — 논문 그림으로 그대로 사용 가능
def reliability_diagram(stats_list, labels, n_bins=15):
    fig, axes = plt.subplots(1, len(stats_list), figsize=(5.2 * len(stats_list), 4.2), squeeze=False)
    for ax, st, lab in zip(axes[0], stats_list, labels):
        xs = [s[0] + 0.5 / n_bins for s in st]; accs = [s[1] for s in st]; confs = [s[2] for s in st]
        ax.plot([0, 1], [0, 1], 'k--', lw=1, label='완벽한 캘리브레이션')
        ax.bar(xs, accs, width=1 / n_bins * .9, alpha=.75, label='정확도', color='#2C5F2D')
        ax.plot(xs, confs, 'o-', color='#D9731A', ms=4, label='평균 신뢰도')
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_xlabel('신뢰도 구간'); ax.set_ylabel('정확도')
        ax.set_title(lab); ax.legend(fontsize=8); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()

reliability_diagram([st_b, st_a],
                    [f'보정 전 (ECE={ece_before:.3f})', f'보정 후 T={T:.2f} (ECE={ece_after:.3f})'])

## 10. 모델 해석 — Grad-CAM (PART 5)

모델이 **종양 영역**을 보고 판단했는지, 아니면 배경·두개골 윤곽 같은 지름길(shortcut)을 학습했는지 확인합니다.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model, self.acts, self.grads = model, None, None
        target_layer.register_forward_hook(lambda m, i, o: setattr(self, 'acts', o))
        target_layer.register_full_backward_hook(lambda m, gi, go: setattr(self, 'grads', go[0]))
    def __call__(self, x, cls=None):
        self.model.eval()
        logits = self.model(x)
        cls = logits.argmax(1) if cls is None else cls
        self.model.zero_grad()
        logits[range(len(x)), cls].sum().backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)             # 채널별 중요도 alpha
        cam = F.relu((w * self.acts).sum(1, keepdim=True))        # (N,1,h,w)
        cam = F.interpolate(cam, x.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam / (cam.amax(dim=(2, 3), keepdim=True) + 1e-8)   # [0,1] 정규화
        return cam.detach(), logits.detach(), cls

def show_gradcam(model, target_layer, df, n=6, device=DEVICE, title='Grad-CAM'):
    model = model.to(device)
    sub = df.sample(n, random_state=SEED)
    xs = torch.stack([eval_tf(Image.open(p).convert('RGB')) for p in sub['path']]).to(device)
    cam, logits, cls = GradCAM(model, target_layer)(xs)
    probs = logits.softmax(1)
    fig, axes = plt.subplots(2, n, figsize=(2.4 * n, 5.2))
    for i in range(n):
        img = inv(xs[i].cpu()).clamp(0, 1).permute(1, 2, 0).numpy()
        axes[0, i].imshow(img); axes[0, i].axis('off')
        axes[0, i].set_title(f"실제 {CLASSES[sub['label'].iloc[i]][:10]}", fontsize=8)
        axes[1, i].imshow(img); axes[1, i].imshow(cam[i, 0].cpu().numpy(), cmap='jet', alpha=.45)
        axes[1, i].axis('off')
        axes[1, i].set_title(f"예측 {CLASSES[cls[i]][:10]}\np={probs[i, cls[i]]:.2f}", fontsize=8)
    plt.suptitle(title); plt.tight_layout(); plt.show()

# ResNet-50 의 마지막 스테이지가 대상 층 (SimpleCNN 이면 model.features[-1])
show_gradcam(tl_model, tl_model.layer4, df_test, n=6, title='ResNet-50 전이학습 — Grad-CAM')

> **읽는 법.** 열지도가 뇌 실질의 병변 부위에 집중되면 바람직한 신호입니다.
> 반대로 영상 가장자리·검은 배경·문자 주석에 집중된다면 데이터 수집 과정의 인공물을 학습한 것이므로,
> 전처리(두개골 제거, 문자 마스킹)를 재검토해야 합니다. Grad-CAM 은 **충실성(faithfulness)이
> 보장되지 않는 사후 설명**이므로, 병변 마스크와의 IoU 같은 정량 지표와 함께 제시하십시오.

## 11. 결과 정리와 과제 (PART 6)

In [ ]:
rows = []
for r in [res_base, res_r18, res_bal, res_tl]:
    rows.append({'모델': r['name'], '정확도': round(r['acc'], 4),
                 'macro-F1': round(r['macro_f1'], 4), 'AUROC': round(r['auroc'], 4)})
summary = pd.DataFrame(rows)
display(summary)
summary.to_csv('results_summary.csv', index=False, encoding='utf-8-sig')
print('results_summary.csv 저장 완료')

plot_history({'SimpleCNN': hist_base, 'ResNet-18(scratch)': hist_r18,
              'SimpleCNN+균형': hist_bal})

### 제출 과제

1. **Ablation 표 완성** — 강의자료 PART 6 의 설계표를 이 데이터셋에 맞춰 채우십시오.
   한 번에 **하나의 변수만** 바꾸고, 같은 예산(에폭·배치)에서 비교합니다.

   | 실험 | 변경점 | 테스트 macro-F1 (seed 3개, mean ± sd) | ECE |
   |---|---|---|---|
   | A | SimpleCNN, 증강 없음, Adam 상수 lr | | |
   | B | A + BatchNorm + AdamW + 코사인 | | |
   | C | B + Crop/Flip/Rotation + 라벨 스무딩 | | |
   | D | C + RandAugment (+ Mixup/CutMix) | | |
   | E | ResNet-18 scratch | | |
   | F | ResNet-50 전이학습 | | |
   | G | ConvNeXt-T 전이학습 | | |

2. **seed 반복** — `SEED` 를 0, 1, 2 로 바꿔 각 실험을 3회 반복하고 평균 ± 표준편차를 보고하십시오.
   *seed 분산보다 작은 차이를 '개선'이라고 주장하지 말 것.*

3. **5-겹 교차검증** — `StratifiedKFold` 로 Training 폴더를 5겹으로 나눠 학습하고,
   테스트 세트 성능의 95% 신뢰구간(부트스트랩 1,000회)을 제시하십시오.

4. **누수 분석 (가장 중요)** — `PROTOCOL` 을 `'original'` / `'repool'` 로 바꿔 **같은 모델·같은 에폭**으로
   평가하고, 테스트 macro-F1 과 클래스별 F1(특히 `no_tumor`)을 표로 비교하십시오.

   | 프로토콜 | 테스트 장수 | macro-F1 | no_tumor F1 |
   |---|---|---|---|
   | original (누수 포함) | 394 | | |
   | repool (누수 제거) | 431 | | |

   두 수치의 차이가 **"이 데이터셋에서 데이터 누수가 만들어낸 허수"** 입니다.
   이 데이터셋으로 얻은 수치를 논문에 쓸 수 있을지, 쓸 수 없다면 무엇이 더 필요한지 논하십시오.

5. **해석** — 오분류된 사례 10장의 Grad-CAM 을 제시하고, 실패 원인을 범주화하십시오
   (병변이 작음 / 클래스 간 형태 유사 / 영상 인공물 / 라벨 오류 등).

**제출물**: 실행 결과가 포함된 `.ipynb` + `results_summary.csv` + 2쪽 보고서(실험 설계·결과·한계).
코드는 시드 고정과 `requirements.txt` 를 포함해야 합니다.

In [ ]:
# 5-겹 교차검증 골격 (과제 3)
from sklearn.model_selection import StratifiedKFold

def cross_validate(model_fn, df_all, n_splits=5, epochs=10, seed=SEED):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    scores = []
    for k, (tr, va) in enumerate(skf.split(df_all, df_all['label']), 1):
        seed_everything(seed + k)
        tl = make_loader(df_all.iloc[tr].reset_index(drop=True), train_tf, shuffle=True)
        vl = make_loader(df_all.iloc[va].reset_index(drop=True), eval_tf)
        m, _, best = train_model(model_fn(), tl, vl, epochs=epochs,
                                 ckpt=f'ckpt_fold{k}.pt', log_every=epochs)
        _, probs, pred, y = predict(m, test_loader)
        mtr = metrics_from(probs.numpy(), pred.numpy(), y.numpy())
        print(f'  fold {k}: val_F1 {best["macro_f1"]:.4f} | test_F1 {mtr["macro_f1"]:.4f}')
        scores.append(mtr)
    s = pd.DataFrame(scores)
    print('\n교차검증 테스트 성능 (mean ± sd)')
    for c in s.columns:
        print(f'  {c:>9}: {s[c].mean():.4f} ± {s[c].std():.4f}')
    return s

# cv_scores = cross_validate(SimpleCNN, df_trainval, n_splits=5, epochs=10)
print('준비 완료 — 위 주석을 해제하면 5-겹 교차검증이 실행됩니다.')

In [ ]:
def bootstrap_ci(y, pred, metric=lambda a, b: f1_score(a, b, average='macro'),
                 n_boot=1000, alpha=0.05, seed=SEED):
    """테스트 세트 부트스트랩 95% 신뢰구간 (과제 3)"""
    rng = np.random.default_rng(seed)
    y, pred = np.asarray(y), np.asarray(pred)
    stats = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2: continue
        stats.append(metric(y[idx], pred[idx]))
    lo, hi = np.percentile(stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(np.mean(stats)), float(lo), float(hi)

mean_f1, lo, hi = bootstrap_ci(res_tl['y'].numpy(), res_tl['pred'].numpy())
print(f"ResNet-50 전이학습 테스트 macro-F1 = {mean_f1:.4f}  (95% CI {lo:.4f} ~ {hi:.4f})")

---

### 참고

* PyTorch 문서 — https://pytorch.org/docs/stable/index.html
* torchvision 모델·가중치 — https://pytorch.org/vision/stable/models.html
* timm (PyTorch Image Models) — https://huggingface.co/docs/timm
* 데이터 출처 — Brain Tumor Classification (MRI), Kaggle (sartajbhuvaji)
* Guo et al. (2017) *On Calibration of Modern Neural Networks*, ICML — temperature scaling
* Selvaraju et al. (2017) *Grad-CAM*, ICCV

**문의**: 최승호 · jcn99250@naver.com